# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library. We will walk through data loading, inspection, extraction, analysis, and visualization, using the Croissant schema and referencing all entities by their `@id`.

### Dataset Source
This dataset is described by a Croissant schema accessible at:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Version: {metadata.version}")
print(f"Published: {metadata.date_published}")
print(f"Keywords: {metadata.keywords}")

## 2. Data Overview
List available record sets, fields, and their `@id`s.

Record sets group related records (rows), and each has a unique `@id` and set of fields/columns. We enumerate all found record sets and their `@id`, and for each field, list their `@id` and name.

In [ ]:
print("Available Record Sets:")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']}, name: {rs['name'] if 'name' in rs else 'N/A'}")
    print("  Fields (columns):")
    if 'field' in rs:
        fields = rs['field']
        if isinstance(fields, dict):
            fields = [fields]
        for field in fields:
            field_id = field['@id']
            field_name = field.get('name', 'N/A')
            print(f"    - Field @id: {field_id}, name: {field_name}")
    print()

## 3. Data Extraction
Load data from chosen record set(s) into DataFrames for analysis. Be sure to reference the record set and field `@id`s as shown above.

Below we will extract data from all available record sets (using their `@id`).

In [ ]:
dataframes = {}
for rs in record_sets:
    record_set_id = rs['@id']
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records from RecordSet {record_set_id}.")
    if not df.empty:
        print(f"Columns for {record_set_id}: {df.columns.tolist()}")
        print(df.head(), "\n")

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing: filter records, normalize numerics, group by attributes.

Select a record set and one or more fields you wish to analyze, referencing their `@id`s as above. For this example, let's select the first non-empty record set with a candidate numeric field (e.g. molecular_weight or abundance, if available).

**Note:** Update `numeric_field_id` and `group_field_id` below to match the actual `@id`s of the appropriate columns in the record set.

In [ ]:
# Example: Select the main protein table for analysis by @id
import numpy as np

# --- Configure these to the actual @id ---
chosen_record_set_id = None
numeric_field_id = None
group_field_id = None

# Try to automatically select a numeric field
for rs in record_sets:
    record_set_id = rs['@id']
    df = dataframes.get(record_set_id)
    if df is not None and not df.empty:
        # Try to select a candidate numeric column
        num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        if num_cols:
            chosen_record_set_id = record_set_id
            numeric_field_id = num_cols[0]  # Just pick the first numeric
            group_field_id = df.columns[0]  # As an example, pick first column as group field
            break

if chosen_record_set_id is None or numeric_field_id is None:
    print("No numeric field found in available record sets.")
else:
    print(f"Analyzing RecordSet: {chosen_record_set_id}")
    print(f"Numeric field for filtering: {numeric_field_id}")
    threshold = df[numeric_field_id].mean()  # Use the mean as threshold
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean): {len(filtered_df)} results.")

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by group_field_id
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped statistics by {group_field_id} (showing first 5 groups):")
        print(grouped_df.head())

## 5. Visualization
Visualize the numeric field distribution and a group comparison for the selected record set and fields.

Below we plot a histogram of the selected numeric field and a bar chart of group means (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if chosen_record_set_id is not None and numeric_field_id is not None:
    # Histogram of numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouped, plot group means (top 10)
    if group_field_id in df.columns:
        group_means = df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)[:10]
        group_means.plot(kind='bar', figsize=(8, 4))
        plt.title(f"Mean {numeric_field_id} by {group_field_id} (top 10)")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion
We have demonstrated how to load and explore a scientific dataset described by a Croissant schema using `mlcroissant`, referencing entities by their `@id`. The workflow included listing record sets and fields, extracting records, performing EDA, and creating visualizations for better understanding the data structure and contents.

Replace field and group selections with specific `@id`s of interest to tailor the notebook further to your analysis goals.